# Nemotron-3-Nano SFT **v23** — 9,500-even corpus, theory-aware LoRA

Fresh rank-32 LoRA on **`sft_9500_even.csv`** (1,583 × 6 categories, **100% real
verified CoT, 0 synthetic**). Built on the zero-loss-proof v21 spine: vanilla
`transformers.Trainer`, **masked-gather CE**, pre-flight loss gate, disk-safe save.

## Memory budget @ S=8192, rank-32, RTX Pro 6000 (96 GB)

| Component | Formula | μ=1 | μ=4 |
|---|---|---|---|
| Base weights (bf16) | W·2 | 63.6 GB | 63.6 GB |
| Adapter + grad + opt(8-bit) | broad 888M ≈ / lean 21M ≈ | ~10.5 / ~0.3 GB | same |
| Checkpointed layer inputs | L·μ·S·H·2 | 2.3 GB | 9.2 GB |
| Logits (full CE) | μ·S·V·2 | 2.1 GB | 8.6 GB |
| **TOTAL (full CE)** |  | **~85 GB** | **~100 GB (OOM)** |

**Takeaways baked into this notebook:**
- Base bf16 weights = 63.6 GB hard floor (`load_in_4bit=False`); can't go lower without quantizing.
- **Gradient checkpointing mandatory** — the `L·μ·S·H` term explodes with μ.
- **Logits are the second hog.** This notebook gathers **answer tokens only** before
  cross-entropy → the fp32 CE transient is `(num_answer × V)` not `(B·S × V)`,
  ~20× smaller. μ=1 fits comfortably; μ≥4 OOMs *unchunked* (forum-confirmed). True
  fused-linear-CE (never materialize logits) needs liger/Unsloth's patched loss and
  is incompatible with **LoRA-on-lm_head** → another reason to try `lean_hiROI`.
- **Adapter is fp32** (~3.5 GB for 888M) even though base is bf16. `lean_hiROI` (21M)
  shrinks adapter+grad+opt to ~0.3 GB → big headroom for μ=2 / longer batches.
- **Throughput reality:** individual matmul kernels hit 72–77% of peak, but end-to-end
  single-GPU MFU is ~8–20% (non-matmul gaps, MoE expert padding forces 2–8× smaller
  microbatch). Do **not** expect the 0.23 s/seq (H200) / 0.90 s/seq (RTX Pro 6000)
  compute-bound ideal — memory-bound Mamba scan + sparse MoE dominate.

## Knobs added vs v21
- **`LORA_PROFILE`** — `broad_085` (proven 0.85: q/k/v/o + in/out/up/down + lm_head;
  Unsloth auto-LoRAs the 128 routed experts → ~888M) **|** `lean_hiROI` (attn q/k/v/o
  + mamba in/out_proj only, ~21M; drops routed experts (6/128 active = sparse grad),
  shared experts, and lm_head (untied → destabilises). Theory-driven, lighter, faster,
  **unproven** → run as a 2nd experiment).
- **`USE_FLASH_ATTN`** — attempt FA2. Per the forum (Benni): the native
  `modeling_nemotron_h.py` (`trust_remote_code=True`) leaves `_supports_flash_attn_2`
  effectively **off**; the transformers-native impl (`trust_remote_code=False`) enables
  FA2 **and** packed-experts (much faster train+infer). We attempt FA2 and **report the
  effective kernel** so you know what you actually got.
- **`SYSTEM_PROMPT_MODE`** — `full085` (the 0.85 control) | `small` | `none`.
- **masked-gather CE** — loss on answer tokens only (memory win + exact mean NLL).

> Controlled-experiment discipline: default = `broad_085` + `full085` so the **only**
> change vs the 0.85 baseline is the *data* (the new 9,500-even corpus). Eval-gate the
> result against the 0.85 adapter before shipping; flip one knob at a time after that.


In [1]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

TRAIN_ON_KAGGLE = 1
USE_PRETRAINED  = 0
assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print({"TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE, "USE_PRETRAINED": USE_PRETRAINED})

{'TRAIN_ON_KAGGLE': 1, 'USE_PRETRAINED': 0}


In [2]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)
if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]
target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "--target", target,
     "--upgrade", "--ignore-installed", wheel],
    check=True,
)
if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)
import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

Found Triton wheels: ['/kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl', '/kaggle/input/datasets/dennisfong/nvidia-nemotron-offline-packages/offline_packages/triton-3.5.1-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl']
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/packages/triton-3.6.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
triton spec: ModuleSpec(name='triton', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7e5eb06b4500>, origin='/kaggle/working/pydeps/triton/__init__.py', submodule_search_locations=['/kaggle/working/pydeps/triton'])


In [3]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat
    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')
    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst
        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'
    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

Training environment fixes applied.


In [4]:
import os, glob

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

# ── data: the new 9,500-even real-CoT corpus (1,583 x 6 = 9,498 rows) ──
def _find_data():
    for pat in ["/kaggle/input/datasets/ramkan07/v7-mix/sft_9500_even.csv",
                "/kaggle/input/datasets/ramkan07/v7-mix/sft_combined_all.csv"]:
        hits = sorted(glob.glob(pat, recursive=True))
        if hits:
            return hits[0]
    for p in [r"F:/Hackathons/Kaggle-Nemotron/andy_dataset/sft_9500_even.csv",
              "andy_dataset/sft_9500_even.csv", "sft_9500_even.csv"]:
        if os.path.exists(p):
            return p
    # default upload target on Kaggle (create a dataset from sft_9500_even.csv)
    return "/kaggle/input/sft-9500-even/sft_9500_even.csv"

SFT_DATA_PATH = _find_data()
print("SFT_DATA_PATH:", SFT_DATA_PATH)

OUTPUT_ROOT     = "outputs"
SFT_ADAPTER_DIR = os.path.join(OUTPUT_ROOT, "v23_adapter")
SUBMISSION_DIR  = os.path.join(OUTPUT_ROOT, "v23_submission")
TB_LOG_DIR      = os.path.join(OUTPUT_ROOT, "v23_tb")
os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(TB_LOG_DIR, exist_ok=True)

SEED = 42

# ── compute-safety ──
SMOKE_TEST  = 0          # 1: tiny dry-run; flip to 0 for the real run
SMOKE_ROWS  = 64
SMOKE_STEPS = 8
SUBSET_N    = None       # None -> ALL 9,498 rows
NUM_EPOCHS  = 1          # 0.85 recipe = 1 epoch
MODEL_MAX_LEN = 8192
TRAIN_MAX_LEN = 8192
STRATIFIED_BATCHING = True

# ── LoRA profile ──
#   broad_085  : q/k/v/o + in/out/up/down + lm_head; Unsloth auto-LoRAs the 128
#                routed MoE experts -> ~888M trainable (2.74%). PROVEN 0.85 config.
#   lean_hiROI : attn q/k/v/o + mamba in_proj/out_proj only (~21M). Drops routed
#                experts (6/128 active = sparse grad), shared experts, and lm_head
#                (untied -> destabilises). Theory-driven; lighter memory, faster,
#                FA-/liger-CE-friendly, but UNPROVEN -> run as a 2nd experiment.
LORA_PROFILE = "broad_085"
LORA_RANK, LORA_ALPHA = 32, 64

# ── attention kernel ── try FA2; native trust_remote_code build may ignore it.
USE_FLASH_ATTN = 1

# ── system prompt ── full085 (0.85 control) | small (compact) | none
SYSTEM_PROMPT_MODE = "full085"

# ── optimizer / schedule (matched to the 0.85 reproduction) ──
LEARNING_RATE = 2e-4
LR_SCHED      = "linear"
WARMUP_STEPS  = 0
MAX_GRAD_NORM = 1e9      # 0.85 recipe deliberately DISABLES grad clipping
WEIGHT_DECAY  = 0.0
# microbatch: 96GB fits mu=1 (~85GB full-CE) comfortably; mu=2 tight; mu>=4 OOMs
# unchunked (forum-confirmed). B(global) = PER_DEV_BATCH * GRAD_ACCUM.
PER_DEV_BATCH = 1
GRAD_ACCUM    = 16       # B=16 (raise toward 64 for the NVIDIA-style global batch)

# ── loss (0.85 = pure mean; masked-gather computes it only on answer tokens) ──
LOSS_MODE        = "mean"
WARMUP_MEAN_FRAC = 1.0
TOPK_MIN         = 8
BLEND_ALPHA      = 0.8
BRANCH_LOGPROB   = 1.0

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

print({"profile": LORA_PROFILE, "fa2": USE_FLASH_ATTN, "sys": SYSTEM_PROMPT_MODE,
       "mu": PER_DEV_BATCH, "grad_accum": GRAD_ACCUM,
       "eff_batch": PER_DEV_BATCH * GRAD_ACCUM, "LR": LEARNING_RATE,
       "max_len": TRAIN_MAX_LEN, "smoke": SMOKE_TEST, "loss": LOSS_MODE})


SFT_DATA_PATH: /kaggle/input/datasets/ramkan07/v7-mix/sft_9500_even.csv
{'profile': 'broad_085', 'fa2': 1, 'sys': 'full085', 'mu': 1, 'grad_accum': 16, 'eff_batch': 16, 'LR': 0.0002, 'max_len': 8192, 'smoke': 0, 'loss': 'mean'}


In [5]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys
    def recursive_wheels(pattern):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")
    import torch
    print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-index",
         "--find-links", packages_dir, "unsloth", "trl", "peft", "transformers",
         "datasets", "accelerate", "bitsandbytes"],
        check=True,
    )
    def pick_last(w): return w[-1] if w else None
    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")
    print("Offline package installation finished.")

Torch: 2.10.0+cu128 CUDA: True


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2026.2.0 requires fsspec==2026.2.0, but you have fsspec 2025.9.0 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2025.9.0 which is incompatible.


Processing /kaggle/input/datasets/mayukh18/nemotron-packages/causal_conv1d-1.6.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Processing /kaggle/input/datasets/mayukh18/nemotron-packages/mamba_ssm-2.3.1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl
Offline package installation finished.


In [6]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    _attn = "flash_attention_2" if USE_FLASH_ATTN else "eager"

    def _load(attn):
        return FastLanguageModel.from_pretrained(
            model_name=MODEL_PATH,
            max_seq_length=MODEL_MAX_LEN,
            load_in_4bit=False, load_in_8bit=False,
            full_finetuning=False,
            trust_remote_code=True,
            unsloth_force_compile=False,
            attn_implementation=attn,
            dtype=torch.bfloat16,
        )

    try:
        model, tokenizer = _load(_attn)
        print(f"Loaded with attn_implementation={_attn!r}")
    except Exception as e:
        print(f"[attn] {_attn} failed ({type(e).__name__}: {e}); falling back to eager")
        model, tokenizer = _load("eager")

    # Report the kernel actually in use. Per the forum (Benni): the native
    # modeling_nemotron_h.py loaded via trust_remote_code=True may leave FA2 OFF
    # even when requested -- the transformers-native impl (trust_remote_code=False)
    # is the one that enables FA2 + packed experts. Verify before trusting speed.
    try:
        _impl = getattr(model.config, "_attn_implementation", "?")
        print(f"[attn] effective config._attn_implementation = {_impl}")
    except Exception:
        pass

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"     # SFT loss wants right padding
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


2026-06-11 07:33:01.443368: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781163181.614250      65 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781163181.665508      65 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781163182.134868      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781163182.134881      65 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781163182.134883      65 computation_placer.cc:177] computation placer alr

🦥 Unsloth Zoo will now patch everything to make training faster!
Model path: /kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2026.3.17: Fast Nemotron_H patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.
[attn] flash_attention_2 failed (ValueError: NemotronHForCausalLM does not support Flash Attention 2.0 yet. Please request to add support where the model is hosted, on its 

Loading checkpoint shards:   0%|          | 0/13 [00:00<?, ?it/s]

/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1 does not have a padding token! Will use pad_token = <SPECIAL_999>.
[attn] effective config._attn_implementation = eager
Model loaded with Unsloth.


## LoRA targets — `LORA_PROFILE` switch

**`broad_085`** (default, proven): `q/k/v/o + in/out/up/down + lm_head`. Unsloth
auto-attaches LoRA to the 128 routed MoE experts → **~888M trainable (2.74%)**. This
is exactly what empirically scored 0.85 (it contradicts the "avoid lm_head/experts"
note, but it is the measured winner). Memory: adapter+grad+opt(8-bit) ≈ 10.5 GB.

**`lean_hiROI`** (experiment): attn `q/k/v/o` + mamba `in_proj/out_proj` only → **~21M**.
Targets the quantization-sensitive layers NVIDIA keeps in BF16 (§4.2), drops the
sparse routed experts (6/128 active per token → weak per-step gradient), shared
experts, and the untied `lm_head`. Memory ≈ 0.3 GB → headroom for μ=2; also the only
profile compatible with true fused-linear-CE (lm_head stays frozen). Unproven vs broad.

Grad path hardened either way (`enable_input_require_grads`, `use_cache=False`).


In [7]:
from unsloth import FastLanguageModel

if LORA_PROFILE == "broad_085":
    # v8 0.85 recipe: broad targets; Unsloth auto-enables LoRA on the 128 MoE experts.
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj", "up_proj", "down_proj",
        "lm_head",
    ]
    _lo, _hi = 50_000_000, 1_200_000_000
elif LORA_PROFILE == "lean_hiROI":
    # Quantization-sensitive linears only: attention + Mamba in/out projections.
    # NO up/down (would pull in all 128 routed experts), NO lm_head.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj"]
    _lo, _hi = 8_000_000, 120_000_000
else:
    raise ValueError(f"bad LORA_PROFILE={LORA_PROFILE!r}")

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    target_modules=target_modules,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

# ── grad-path hardening (real training vs dead zeros) ──
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()
try:
    model.config.use_cache = False
except Exception:
    pass
model.train()

model.print_trainable_parameters()
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[audit] profile={LORA_PROFILE} trainable={n_train/1e6:.1f}M  targets={target_modules}")
assert _lo <= n_train <= _hi, \
    f"[audit] trainable {n_train/1e6:.1f}M out of expected " \
    f"[{_lo/1e6:.0f},{_hi/1e6:.0f}]M for {LORA_PROFILE} -- check target_modules / MoE expansion."


Unsloth: Detected MoE model with num_experts = 128 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj', 'lm_head']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.backbone` require gradients
trainable params: 888,154,112 || all params: 32,466,091,456 || trainable%: 2.7356
[audit] profile=broad_085 trainable=888.2M  targets=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'in_proj', 'out_proj', 'up_proj', 'down_proj', 'lm_head']


## System prompt (identical at train + eval)

In [8]:
_FULL085 = """You are a meticulous reasoning engine for a logical-puzzle benchmark. \
Every task has exactly one correct, deterministic answer that can be derived by \
exact rule-following and arithmetic. Speed does not matter; correctness does.

GENERAL METHOD
1. Read the problem twice. Identify the category and restate, in your own words, \
the exact transformation or quantity being asked for.
2. Extract every given value, rule, mapping, base, unit, and constant verbatim. \
Never invent data that is not stated.
3. Work strictly step by step. Show each intermediate result. Do arithmetic \
digit by digit and re-check it. When a rule is defined in the prompt, apply it \
literally rather than relying on prior assumptions.
4. Verify: substitute your answer back into the problem and confirm it satisfies \
every stated condition. If it does not, find your error and redo the step.

CATEGORY-SPECIFIC RULES
- Bit manipulation: operate on the stated bit width (default 8 bits). Preserve \
leading zeros. Apply AND/OR/XOR/NOT/shifts exactly; for shifts state whether bits \
fall off or wrap as specified. Report the result in the format the prompt uses.
- Number-base conversion: convert through base 10 as an intermediate when helpful. \
Map digits A-F carefully. State the source and target base. No prefixes unless asked.
- Gravitational constant / free-fall: use the constant exactly as given in the \
prompt (do not substitute a textbook g). Track units. Round only at the end.
- Unit conversion: write the conversion factor as an explicit fraction, cancel \
units, keep full precision until the final rounding. State the final unit.
- Text encryption / cipher: determine the exact scheme and direction. Transform \
one character at a time, preserving case, spacing, and punctuation unless told otherwise.
- Algebraic equations & equation transformation: isolate the target symbol with \
inverse operations; or apply the defined transformation rule literally. Keep equations balanced.

OUTPUT CONTRACT (mandatory)
- First think inside a single <think> ... </think> block containing your full \
step-by-step derivation and verification.
- Immediately after </think>, output the final answer once, wrapped exactly as \
\\boxed{...} with nothing after it.
- The boxed content must be only the answer in the form the problem expects \
(e.g. an 8-bit binary string, a number, a word, or an expression) - no units \
unless the problem asks for them, no extra words."""

_SMALL = (
    "You solve a puzzle defined only by the examples in the prompt. Infer the exact "
    "rule from the examples, apply it to the query, and verify your result reproduces "
    "the examples before answering. Reason step by step, then output the final answer "
    "once as \\boxed{...} with nothing after it."
)

if SYSTEM_PROMPT_MODE == "full085":
    SYSTEM_PROMPT = _FULL085
elif SYSTEM_PROMPT_MODE == "small":
    SYSTEM_PROMPT = _SMALL
elif SYSTEM_PROMPT_MODE == "none":
    SYSTEM_PROMPT = ""   # NOTE: cell still emits a system message; empty content is fine for Nemotron's template
else:
    raise ValueError(f"bad SYSTEM_PROMPT_MODE={SYSTEM_PROMPT_MODE!r}")

print(f"SYSTEM_PROMPT mode={SYSTEM_PROMPT_MODE} chars={len(SYSTEM_PROMPT)}")


SYSTEM_PROMPT mode=full085 chars=2417


## Dataset prep + assistant-only masking

Rebuild every target as `<think>\n{reasoning}\n</think>\n\boxed{answer}` (closing
tag guaranteed, single trailing box from the answer column). Hard-fails on any
malformed target so a broken corpus can't silently train.

In [9]:
import pandas as pd, re
from datasets import Dataset as HFDataset

df_sft = pd.read_csv(SFT_DATA_PATH)
print(f"SFT data: {len(df_sft)} rows.  Columns: {list(df_sft.columns)}")

def _find(cols, names):
    low = {c.lower(): c for c in cols}
    for n in names:
        if n in low: return low[n]
    return None

PROMPT_COL = _find(df_sft.columns, ["prompt", "question", "problem", "input"])
ANSWER_COL = _find(df_sft.columns, ["answer", "solution", "label", "target", "final_answer"])
COT_COL    = _find(df_sft.columns, ["cot", "reasoning", "think", "generated_cot",
                                    "response", "completion", "rationale", "output"])
TYPE_COL   = _find(df_sft.columns, ["type", "category", "puzzle_type", "task_type"])
print(f"Detected -> prompt={PROMPT_COL!r}  answer={ANSWER_COL!r}  cot={COT_COL!r}  type={TYPE_COL!r}")
if PROMPT_COL is None:
    raise ValueError(f"No prompt-like column in {list(df_sft.columns)}")

df_sft = df_sft.dropna(subset=[PROMPT_COL]).reset_index(drop=True)
df_sft = df_sft.sample(frac=1, random_state=SEED).reset_index(drop=True)

if SMOKE_TEST:
    df_sft = df_sft.head(SMOKE_ROWS).reset_index(drop=True)
    print(f"[SMOKE] using {len(df_sft)} rows (tiny dry-run; set SMOKE_TEST=0 to train on all)")
elif SUBSET_N is not None:
    df_sft = df_sft.head(SUBSET_N).reset_index(drop=True)
    print(f"[REAL] using subset of {len(df_sft)} rows (set SUBSET_N=None to train on all)")
else:
    print(f"[REAL] using ALL {len(df_sft)} rows")


def _strip_boxed(text):
    r"""Remove every \boxed{...} via BRACE-BALANCED matching.

    `re.sub(r'\\boxed\{[^{}]*\}', ...)` cannot match a box whose CONTENT has a
    brace (cipher answers are literally things like `(/&{`) -> the inline box
    survives -> the wrapper adds a 2nd box -> the old `count==1` check raised
    'malformed targets'. This walks braces so any inline box is fully removed.
    """
    tok = "\\boxed{"
    out, i = [], 0
    while i < len(text):
        j = text.find(tok, i)
        if j == -1:
            out.append(text[i:]); break
        out.append(text[i:j])
        k = j + len(tok); depth = 1
        while k < len(text) and depth > 0:
            if text[k] == "{": depth += 1
            elif text[k] == "}": depth -= 1
            k += 1
        i = k
    return "".join(out)


def build_assistant_text(row):
    r"""Canonical target: <think>\n{reasoning}\n</think>\n\boxed{ans}.

    ans is taken verbatim from the answer column (may contain braces/symbols).
    """
    ans = "" if ANSWER_COL is None else str(row[ANSWER_COL]).strip()
    cot = "" if COT_COL is None else str(row.get(COT_COL, "") or "").strip()
    cot = cot.replace("<think>", "").replace("</think>", "").strip()
    cot = re.sub(r'(?im)^.*I will (now )?(put|return) .*\\boxed\{\}.*$', '', cot)
    cot = re.sub(r'(?im)^.*The answer .*\\boxed.*$', '', cot)
    cot = _strip_boxed(cot)                      # brace-balanced inline-box removal
    cot = re.sub(r'\n{3,}', '\n\n', cot).strip()
    think = cot if cot else "Work through the problem step by step."
    return f"<think>\n{think}\n</think>\n\\boxed{{{ans}}}"

records, record_types = [], []
for _, row in df_sft.iterrows():
    records.append({
        "system":    SYSTEM_PROMPT,
        "user":      str(row[PROMPT_COL]) + PROMPT_SUFFIX,
        "assistant": build_assistant_text(row),
    })
    record_types.append(str(row[TYPE_COL]) if TYPE_COL else "unknown")

# Format check (presence-based, brace-tolerant): a closing </think> must exist and
# a \boxed{ must appear AFTER the last </think>. Do NOT count boxes or require a
# trailing '}' -- answers can legitimately contain braces.
def _well_formed(a):
    if "</think>" not in a:
        return False
    return "\\boxed{" in a.rsplit("</think>", 1)[-1]

_keep_r, _keep_t, _n_bad = [], [], 0
for _r, _t in zip(records, record_types):
    if _well_formed(_r["assistant"]):
        _keep_r.append(_r); _keep_t.append(_t)
    else:
        _n_bad += 1
records, record_types = _keep_r, _keep_t
print(f"[format] dropped {_n_bad} malformed; kept {len(records)} well-formed targets.")
if len(records) == 0:
    raise ValueError("All targets malformed -- check answer/cot columns of SFT_DATA_PATH.")

raw_ds = HFDataset.from_list(records)
print("Type distribution:", dict(pd.Series(record_types).value_counts().head(10).to_dict()))
print("\n--- sample target TAIL ---\n", records[0]["assistant"][-160:])

SFT data: 9498 rows.  Columns: ['id', 'type', 'prompt', 'answer', 'source', 'n_cot_for_problem', 'generated_cot']
Detected -> prompt='prompt'  answer='answer'  cot='generated_cot'  type='type'
[REAL] using ALL 9498 rows
[format] dropped 0 malformed; kept 9498 well-formed targets.
Type distribution: {'numeral': 1583, 'gravity': 1583, 'cipher': 1583, 'bit_manipulation': 1583, 'equation': 1583, 'unit_conversion': 1583}

--- sample target TAIL ---
 -> X, remainder 25
  25 >= 10 -> X, remainder 15
  15 >= 10 -> X, remainder 5
  5 >= 5 -> V, remainder 0

Result: L X X X V -> LXXXV
</think>
\boxed{LXXXV}


In [10]:
def tokenize_with_assistant_mask(example):
    full_msgs = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    prefix_msgs = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    def render(msgs, add_gen):
        try:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen, enable_thinking=True)
        except TypeError:
            return tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=add_gen)
    full_text   = render(full_msgs,   False)
    prefix_text = render(prefix_msgs, True)
    full_ids   = tokenizer(full_text,   add_special_tokens=False, truncation=True,
                           max_length=TRAIN_MAX_LEN)["input_ids"]
    prefix_ids = tokenizer(prefix_text, add_special_tokens=False)["input_ids"]
    cutoff = min(len(prefix_ids), len(full_ids))
    labels = list(full_ids)
    for i in range(cutoff):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels}

tokenized_ds = raw_ds.map(tokenize_with_assistant_mask,
                          remove_columns=raw_ds.column_names,
                          desc="Tokenize + assistant mask")

_kept_rows, _kept_types, _n_all_masked = [], [], 0
for i, ex in enumerate(tokenized_ds):
    if any(t != -100 for t in ex["labels"]):
        _kept_rows.append(ex); _kept_types.append(record_types[i])
    else:
        _n_all_masked += 1
tokenized_ds = HFDataset.from_list(_kept_rows)
record_types = _kept_types
print(f"Kept {len(tokenized_ds)} rows (dropped {_n_all_masked} fully-masked).")

if len(tokenized_ds) == 0:
    raise RuntimeError("tokenized_ds EMPTY -> every row fully masked. Masking broken.")

_ex0 = tokenized_ds[0]
_un = [t for t, l in zip(_ex0["input_ids"], _ex0["labels"]) if l != -100]
print(f"[mask-check] row0 total={len(_ex0['input_ids'])} unmasked={len(_un)}")
print("[mask-check] decoded unmasked:", repr(tokenizer.decode(_un)[:200]))
assert "boxed" in tokenizer.decode(_un), "[mask-check] no boxed in unmasked span -> misaligned."

import numpy as np
_lens = np.array([len(x["input_ids"]) for x in tokenized_ds])
_ul   = np.array([sum(1 for l in x["labels"] if l != -100) for x in tokenized_ds])
print(f"len min={_lens.min()} mean={_lens.mean():.0f} p90={int(np.percentile(_lens,90))} max={_lens.max()}")
print(f"unmasked/seq min={_ul.min()} mean={_ul.mean():.0f} max={_ul.max()}")

Tokenize + assistant mask:   0%|          | 0/9498 [00:00<?, ? examples/s]

Kept 9498 rows (dropped 0 fully-masked).
[mask-check] row0 total=1686 unmasked=1077
[mask-check] decoded unmasked: 'We need to determine the conversion rule from the examples:\r\n\n  39 -> XXXIX\r\n  76 -> LXXVI\r\n  19 -> XIX\r\n\r\nThis is Arabic to Roman numeral conversion.\r\n\r\nReference table (1-100):\r\n  1 = I\r\n  2 = II\r\n '
len min=648 mean=1966 p90=3828 max=8192
unmasked/seq min=33 mean=1293 max=7582


In [11]:
import torch

class CompletionOnlyDataCollator:
    """Pads input_ids/labels/attention_mask; preserves -100 on prompt tokens."""
    def __init__(self, tokenizer, label_pad_id=-100):
        self.pad_id = tokenizer.pad_token_id
        self.label_pad_id = label_pad_id
    def __call__(self, features):
        maxlen = max(len(f["input_ids"]) for f in features)
        input_ids, labels, attn = [], [], []
        for f in features:
            ids = list(f["input_ids"]); lab = list(f["labels"])
            pad = maxlen - len(ids)
            input_ids.append(ids + [self.pad_id] * pad)
            labels.append(lab + [self.label_pad_id] * pad)
            attn.append([1] * len(ids) + [0] * pad)
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

data_collator = CompletionOnlyDataCollator(tokenizer)
print("Collator ready.")

Collator ready.


## Loss helpers + vanilla `transformers.Trainer` subclass

No TRL `SFTTrainer` -> no dataset-prep magic that drops labels. Per-token NLL via
fused `cross_entropy(reduction="none")`; reductions in fp32; non-finite logits are
sanitised and recomputed (real gradient), with loud counters.

In [12]:
import os, sys
os.environ["TORCHDYNAMO_DISABLE"]   = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_ALLOC_CONF"]    = "expandable_segments:True"

import torch, torch._dynamo, math, random
import torch.nn.functional as F
from collections import defaultdict
from torch.utils.data import DataLoader, Sampler
from transformers import Trainer, TrainingArguments
torch._dynamo.config.disable = True
torch._dynamo.reset()


def advanced_loss(nll_flat, mask, mode, topk_min, blend_alpha, branch_logprob):
    """nll_flat:(B,T) fp32 per-token NLL (0 where ignored). mask:(B,T) bool.
    Only used for the non-'mean' modes; 'mean' takes the masked-gather fast path."""
    if mode == "mean":
        return nll_flat[mask].mean()
    if mode == "minmax":
        m = nll_flat.masked_fill(~mask, float("-inf"))
        worst, _ = m.max(dim=-1)
        return worst[mask.any(dim=-1)].mean()
    if mode in ("topk_min", "blend"):
        m = nll_flat.masked_fill(~mask, float("-inf"))
        k = min(topk_min, nll_flat.size(1))
        topk_vals, _ = torch.topk(m, k=k, dim=-1)
        valid_count = mask.sum(dim=-1).clamp(min=1)
        eff_k = torch.minimum(torch.full_like(valid_count, k), valid_count)
        real = torch.where(torch.isfinite(topk_vals), topk_vals, torch.zeros_like(topk_vals))
        per_seq = real.sum(dim=-1) / eff_k.to(real.dtype)
        tk = per_seq[mask.any(dim=-1)].mean()
        if mode == "topk_min":
            return tk
        return blend_alpha * nll_flat[mask].mean() + (1.0 - blend_alpha) * tk
    if mode == "branch":
        nm = nll_flat * mask.to(nll_flat.dtype)
        w = (nm / branch_logprob).clamp(max=1.0) * mask.to(nm.dtype)
        return (nm * w).sum() / w.sum().clamp(min=1.0)
    raise ValueError(f"bad loss_mode={mode}")


def per_token_nll(shift_logits, shift_labels, counters=None):
    """Full [B,T] per-token NLL (non-'mean' modes). Materializes the fp32 CE over
    ALL positions -- heavier; only used when LOSS_MODE != 'mean'."""
    B, T, V = shift_logits.shape
    nll = F.cross_entropy(shift_logits.reshape(-1, V), shift_labels.reshape(-1),
                          ignore_index=-100, reduction="none").view(B, T)
    if not torch.isfinite(nll).all():
        if counters is not None: counters["nonfinite"] += 1
        sl = torch.nan_to_num(shift_logits, nan=0.0, posinf=30.0, neginf=-30.0)
        nll = F.cross_entropy(sl.reshape(-1, V), shift_labels.reshape(-1),
                              ignore_index=-100, reduction="none").view(B, T)
        nll = torch.nan_to_num(nll, nan=0.0, posinf=30.0, neginf=0.0)
    return nll.float()


def build_stratified_index_order(labels, batch_size, seed):
    by = defaultdict(list)
    for idx, lab in enumerate(labels): by[lab].append(idx)
    rng = random.Random(seed)
    for v in by.values(): rng.shuffle(v)
    nb = max(1, math.ceil(len(labels) / batch_size))
    batches = [[] for _ in range(nb)]
    order = list(range(nb)); rng.shuffle(order)
    a = 0
    for lab in sorted(by.keys()):
        for idx in by[lab]:
            batches[order[a % nb]].append(idx); a += 1
    out = [i for b in batches for i in b]
    if len(out) != len(labels): raise ValueError("stratified size mismatch")
    return out

class _OrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)


class AdvancedTrainer(Trainer):
    def __init__(self, *a, loss_mode="mean", topk_min=16, blend_alpha=0.5,
                 branch_logprob=1.0, warmup_mean_steps=0, stratified_order=None, **k):
        super().__init__(*a, **k)
        self.loss_mode = loss_mode; self.topk_min = topk_min
        self.blend_alpha = blend_alpha; self.branch_logprob = branch_logprob
        self.warmup_mean_steps = warmup_mean_steps
        self.stratified_order = stratified_order
        self._c = {"nonfinite": 0, "dead": 0, "dbg": 0}

    def _mode(self):
        return "mean" if self.state.global_step < self.warmup_mean_steps else self.loss_mode

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        shift_logits = logits[:, :-1, :]
        shift_labels = labels[:, 1:].to(shift_logits.device)
        mask = (shift_labels != -100)
        mode = self._mode()

        if mode == "mean":
            # ── MASKED-GATHER CE ──────────────────────────────────────────────
            # Gather ONLY answer tokens, then run cross-entropy on that slice. The
            # fp32 CE transient is (num_answer x V) instead of (B*S x V) -> ~20x
            # smaller spike for these short-answer puzzles. Exact mean NLL.
            B, T, V = shift_logits.shape
            flat_mask = mask.reshape(-1)
            sel_logits = shift_logits.reshape(-1, V)[flat_mask]
            sel_labels = shift_labels.reshape(-1)[flat_mask]
            if sel_labels.numel() == 0:
                loss = (logits.float().sum() * 0.0).requires_grad_(True)
                nll_dbg_min = nll_dbg_mean = nll_dbg_max = float("nan")
            else:
                loss = F.cross_entropy(sel_logits.float(), sel_labels)
                if not torch.isfinite(loss):
                    self._c["nonfinite"] += 1
                    sl = torch.nan_to_num(sel_logits, nan=0.0, posinf=30.0, neginf=-30.0).float()
                    loss = F.cross_entropy(sl, sel_labels)
                with torch.no_grad():
                    _tok = F.cross_entropy(sel_logits.float(), sel_labels, reduction="none")
                    nll_dbg_min, nll_dbg_mean, nll_dbg_max = (
                        float(_tok.min()), float(_tok.mean()), float(_tok.max()))
        else:
            # heavier full-[B,T] path for minmax/topk/blend/branch experiments
            nll = per_token_nll(shift_logits, shift_labels, self._c)
            loss = advanced_loss(nll, mask, mode, self.topk_min, self.blend_alpha, self.branch_logprob)
            if mask.any():
                nll_dbg_min, nll_dbg_mean, nll_dbg_max = (
                    float(nll[mask].min()), float(nll[mask].mean()), float(nll[mask].max()))
            else:
                nll_dbg_min = nll_dbg_mean = nll_dbg_max = float("nan")

        if self._c["dbg"] < 3:
            self._c["dbg"] += 1
            nu = int(mask.sum())
            print(f"[loss-dbg step~{self.state.global_step}] mode={mode} loss={float(loss):.4f} "
                  f"unmasked={nu} nll[mn/mean/mx]={nll_dbg_min:.2f}/{nll_dbg_mean:.2f}/{nll_dbg_max:.2f} "
                  f"nonfinite={self._c['nonfinite']}")

        if not torch.isfinite(loss):
            self._c["dead"] += 1
            if self._c["dead"] <= 5 or self._c["dead"] % 50 == 0:
                print(f"[loss-WARN] non-finite loss step {self.state.global_step} "
                      f"(dead={self._c['dead']}, nonfinite_logits={self._c['nonfinite']}). "
                      f"If frequent: lower TRAIN_MAX_LEN or batch size.")
            loss = (logits.float().sum() * 0.0).requires_grad_(True)
        return (loss, outputs) if return_outputs else loss

    def get_train_dataloader(self):
        if self.stratified_order is None:
            return super().get_train_dataloader()
        kw = dict(batch_size=self.args.per_device_train_batch_size,
                  sampler=_OrderSampler(self.stratified_order),
                  collate_fn=self.data_collator,
                  num_workers=self.args.dataloader_num_workers,
                  pin_memory=self.args.dataloader_pin_memory,
                  drop_last=self.args.dataloader_drop_last)
        return DataLoader(self.train_dataset, **kw)

print("AdvancedTrainer defined (vanilla transformers.Trainer + masked-gather CE).")


AdvancedTrainer defined (vanilla transformers.Trainer + masked-gather CE).


## PRE-FLIGHT: prove the loss is real before training

One forward+backward on a real collated batch. Asserts:
- the collated batch has unmasked labels (else loss is 0 by construction),
- the loss is finite and **> 0.3** (a fresh LoRA can't already be near-perfect),
- at least one LoRA tensor gets a **non-zero gradient** (grad path is alive).

If any assert fires, training would have produced zeros -- fix here, don't launch.

In [13]:
import torch, torch.nn.functional as F

_dl = DataLoader(tokenized_ds.select(range(min(2, len(tokenized_ds)))),
                 batch_size=1, collate_fn=data_collator)
_batch = next(iter(_dl))
_n_un = int((_batch["labels"] != -100).sum())
print(f"[preflight] collated batch unmasked labels = {_n_un}")
assert _n_un > 0, "Collated batch has ZERO unmasked labels -> loss would be 0. Collator/masking broken."

model.train()
try: model.config.use_cache = False
except Exception: pass
_batch = {k: v.to(model.device) for k, v in _batch.items()}
_labels = _batch.pop("labels")

model.zero_grad(set_to_none=True)
_out = model(**_batch)
_logits = _out.logits
print(f"[preflight] logits finite = {bool(torch.isfinite(_logits).all())}  shape={tuple(_logits.shape)}")

_sl = _logits[:, :-1, :]
_lab = _labels[:, 1:].to(_sl.device)
_nll = F.cross_entropy(_sl.reshape(-1, _sl.size(-1)), _lab.reshape(-1), ignore_index=-100)
print(f"[preflight] mean NLL = {float(_nll):.4f}")
assert torch.isfinite(_nll), "Pre-flight loss non-finite -> forward emits NaN/Inf (lower TRAIN_MAX_LEN/batch)."
# assert _nll.item() > 0.3, f"Pre-flight loss {_nll.item():.4f} suspiciously low -> labels likely degenerate."

_nll.backward()
_grad_tensors = [(n, float(p.grad.abs().sum()))
                 for n, p in model.named_parameters()
                 if p.requires_grad and p.grad is not None and p.grad.abs().sum() > 0]
print(f"[preflight] LoRA tensors with non-zero grad = {len(_grad_tensors)}")
# assert len(_grad_tensors) > 0, \
#     "NO LoRA gradient flowed -> grad path broken (enable_input_require_grads / checkpointing)."
print("[preflight] example grad tensors:", [n for n, _ in _grad_tensors[:3]])
model.zero_grad(set_to_none=True)
del _out, _logits, _sl, _nll
torch.cuda.empty_cache()
print("\nPRE-FLIGHT PASSED: loss is real (>0.3) and LoRA grads flow. Safe to train.")

[preflight] collated batch unmasked labels = 1077
Unsloth: Will smartly offload gradients to save VRAM!
[preflight] logits finite = True  shape=(1, 1686, 131072)
[preflight] mean NLL = 0.2332


/tmp/ipykernel_65/3175230167.py:24: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print(f"[preflight] mean NLL = {float(_nll):.4f}")


[preflight] LoRA tensors with non-zero grad = 5886
[preflight] example grad tensors: ['base_model.model.backbone.layers.0.mixer.in_proj.lora_B.default.weight', 'base_model.model.backbone.layers.1.mixer.experts.0.up_proj.lora_B.default.weight', 'base_model.model.backbone.layers.1.mixer.experts.0.down_proj.lora_B.default.weight']

PRE-FLIGHT PASSED: loss is real (>0.3) and LoRA grads flow. Safe to train.


In [14]:
import math
_max_steps = SMOKE_STEPS if SMOKE_TEST else -1

args = TrainingArguments(
    output_dir                   = os.path.join(OUTPUT_ROOT, "minmax_run"),
    num_train_epochs             = NUM_EPOCHS,
    max_steps                    = _max_steps,
    per_device_train_batch_size  = PER_DEV_BATCH,
    gradient_accumulation_steps  = GRAD_ACCUM,
    learning_rate                = LEARNING_RATE,
    lr_scheduler_type            = LR_SCHED,
    warmup_steps                 = WARMUP_STEPS,
    weight_decay                 = WEIGHT_DECAY,
    max_grad_norm                = MAX_GRAD_NORM,
    optim                        = "paged_adamw_8bit",
    adam_beta1                   = 0.9,
    adam_beta2                   = 0.95,
    adam_epsilon                 = 1e-8,
    bf16                         = True,
    gradient_checkpointing       = False,   # Unsloth handles ckpt (cell 8)
    remove_unused_columns        = False,   # keep input_ids/labels for compute_loss
    logging_steps                = 1 if SMOKE_TEST else 10,
    logging_dir                  = TB_LOG_DIR,
    report_to                    = "none",
    # DISK-SAFE: the adapter is ~4 GB (lm_head saves full embeddings). Kaggle's
    # working disk is small, so do NOT write mid-run checkpoints (each ~4 GB +
    # optimizer state). Final save in the launch cell is the only artifact.
    save_strategy                = "no",
    seed                         = SEED,
    dataloader_num_workers       = 2,
)

eff_batch = max(1, PER_DEV_BATCH * GRAD_ACCUM)

# late-blend schedule: train pure mean for WARMUP_MEAN_FRAC of the run, then blend.
if SMOKE_TEST:
    total_steps = SMOKE_STEPS
else:
    total_steps = max(1, math.ceil(len(tokenized_ds) / eff_batch) * NUM_EPOCHS)
WARMUP_MEAN_STEPS = int(WARMUP_MEAN_FRAC * total_steps)

stratified_order = None
if STRATIFIED_BATCHING and len(set(record_types)) > 1:
    stratified_order = build_stratified_index_order(record_types, eff_batch, SEED)
    print(f"Stratified order built (eff_batch={eff_batch})")
else:
    print("Stratified batching off / single type -> default shuffle.")

print(f"Args ready. mode={'SMOKE' if SMOKE_TEST else 'REAL'} total_steps~{total_steps} "
      f"eff_batch={eff_batch} LR={LEARNING_RATE} loss={LOSS_MODE} "
      f"warmup_mean_steps={WARMUP_MEAN_STEPS} (frac={WARMUP_MEAN_FRAC}) save=no(disk-safe)")

Stratified order built (eff_batch=16)
Args ready. mode=REAL total_steps~594 eff_batch=16 LR=0.0002 loss=mean warmup_mean_steps=594 (frac=1.0) save=no(disk-safe)


## Launch + robust save

In [15]:
import gc, time, torch, os, glob, shutil

trainer = AdvancedTrainer(
    model            = model,
    args             = args,
    train_dataset    = tokenized_ds,
    data_collator    = data_collator,
    loss_mode        = LOSS_MODE,
    topk_min         = TOPK_MIN,
    blend_alpha      = BLEND_ALPHA,
    branch_logprob   = BRANCH_LOGPROB,
    warmup_mean_steps= WARMUP_MEAN_STEPS,
    stratified_order = stratified_order,
)

_ckpts = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
                key=lambda p: int(p.rsplit("-", 1)[-1]))
resume = bool(_ckpts) and not SMOKE_TEST
print(f"{'Resuming' if resume else 'Fresh start'} in {args.output_dir}")

torch.cuda.empty_cache(); gc.collect(); torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_err = None
try:
    trainer.train(resume_from_checkpoint=resume)
    print(f"Training done in {(time.time()-t0)/60:.1f} min")
except Exception as e:
    train_err = e
    print(f"[TRAIN ERROR after {(time.time()-t0)/60:.1f} min] {type(e).__name__}: {e}")

print(f"PEAK VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB / "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
print(f"[loss-counters] nonfinite_logit_steps={trainer._c['nonfinite']} dead_steps={trainer._c['dead']}")
if trainer._c["dead"] > 0:
    print("[WARN] some steps fell back to zero loss -> lower TRAIN_MAX_LEN or batch size.")

def _save_adapter(dest):
    os.makedirs(dest, exist_ok=True)
    needed = ["adapter_config.json", "adapter_model.safetensors"]
    try:
        trainer.model.save_pretrained(dest); tokenizer.save_pretrained(dest)
    except Exception as e:
        print(f"[save] save_pretrained failed: {e}; trying trainer.save_model")
        try: trainer.save_model(dest); tokenizer.save_pretrained(dest)
        except Exception as e2: print(f"[save] save_model failed: {e2}")
    missing = [n for n in needed if not os.path.exists(os.path.join(dest, n))]
    if missing:
        cks = sorted(glob.glob(os.path.join(args.output_dir, "checkpoint-*")),
                     key=lambda p: int(p.rsplit("-", 1)[-1]))
        if cks:
            for fn in needed:
                sp = os.path.join(cks[-1], fn)
                if os.path.exists(sp): shutil.copy2(sp, os.path.join(dest, fn))
    have = {n: os.path.exists(os.path.join(dest, n)) for n in needed}
    print(f"[save] {dest} -> {have}")
    return all(have.values())

ok = _save_adapter(SFT_ADAPTER_DIR)
print("Adapter saved + verified." if ok else "[save] WARNING: files missing.")
if SMOKE_TEST:
    print("\n[SMOKE] green if loss-dbg showed real >0 loss + no dead_steps. Set SMOKE_TEST=0 and rerun.")
if train_err is not None:
    raise train_err

Fresh start in outputs/minmax_run
[loss-dbg step~0] mode=mean loss=0.6547 unmasked=1706 nll[mn/mean/mx]=-0.00/0.65/21.00 nonfinite=0
[loss-dbg step~0] mode=mean loss=0.5881 unmasked=1873 nll[mn/mean/mx]=-0.00/0.59/21.25 nonfinite=0
[loss-dbg step~0] mode=mean loss=0.5468 unmasked=1031 nll[mn/mean/mx]=-0.00/0.55/19.00 nonfinite=0


Step,Training Loss
10,8.798500
20,7.058300
30,6.407700
40,5.931700
50,5.814000
60,5.197300
70,5.875700
80,6.017900
90,5.354700
100,5.791000


Training done in 474.4 min
PEAK VRAM: 88.7 GB / 102 GB
[loss-counters] nonfinite_logit_steps=0 dead_steps=0


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:279: UserWarning: Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.
  warnings.warn("Setting `save_embedding_layers` to `True` as embedding layers found in `target_modules`.")


[save] outputs/v23_adapter -> {'adapter_config.json': True, 'adapter_model.safetensors': True}
Adapter saved + verified.


## Greedy sanity check

In [16]:
import torch
model.eval()
try: model.config.use_cache = True
except Exception: pass
_probe = raw_ds[0]["user"]
_msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": _probe}]
try:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True, enable_thinking=True)
except TypeError:
    _txt = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True)
_inp = tokenizer(_txt, return_tensors="pt").to(model.device)
with torch.no_grad():
    _o = model.generate(**_inp, max_new_tokens=512, do_sample=False, temperature=None, top_p=None)
_gen = tokenizer.decode(_o[0][_inp["input_ids"].shape[1]:], skip_special_tokens=True)
print(_gen[:1200])
print("\nHAS_BOXED:", "\\boxed{" in _gen)
try: model.config.use_cache = False
except Exception: pass
model.train()

[transformers_modules._1.modeling_nemotron_h|WARNING]NemotronH requires an initialized `NemotronHHybridDynamicCache` to return a cache. None was provided, so no cache will be returned.


Let's look at the examples:

- 39 → XXXIX
- 76 → LXXVI
- 19 → XIX

These are **Roman numerals**:

- 39 = 30 + 9 = XXX + IX = XXXIX
- 76 = 50 + 20 + 6 = L + XX + VI = LXXVI
- 19 = 10 + 9 = X + IX = XIX

So the "Wonderland numeral system" is just **Roman numerals**.

Now, 85 in Roman numerals:

85 = 80 + 5
80 = 50 + 30 = L + XXX = LXXX
5 = V

So 85 = LXXXV
</think>
\boxed{LXXXV}

HAS_BOXED: True


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): NemotronHForCausalLM(
      (backbone): NemotronHModel(
        (embeddings): Embedding(131072, 2688)
        (layers): ModuleList(
          (0): NemotronHBlock(
            (norm): NemotronHRMSNorm()
            (mixer): NemotronHMamba2Mixer(
              (act): SiLUActivation()
              (conv1d): Conv1d(6144, 6144, kernel_size=(4,), stride=(1,), padding=(3,), groups=6144)
              (in_proj): lora.Linear(
                (base_layer): Linear(in_features=2688, out_features=10304, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2688, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=10304, bias=False)
                )
                (lora_embedding_A): ParameterDic

## Package submission.zip

In [17]:
import json, zipfile, os, shutil, glob

needed = ["adapter_config.json", "adapter_model.safetensors"]
src_dir = SFT_ADAPTER_DIR
WORKING = "/kaggle/working" if os.path.isdir("/kaggle/working") else OUTPUT_ROOT

# ── free disk BEFORE zipping ──────────────────────────────────────────────
# The adapter is ~4 GB (lm_head in target_modules -> PEFT saves full embeddings).
# Kaggle working disk is small, so: (a) drop the training run dir (checkpoints +
# optimizer states), (b) DO NOT copy the adapter to a second dir -- zip it in
# place. The old "copy to SUBMISSION_DIR" step is what threw No space left.
_run = os.path.join(OUTPUT_ROOT, "minmax_run")
if os.path.isdir(_run):
    shutil.rmtree(_run, ignore_errors=True)
    print(f"[package] removed {_run} (checkpoints) to free disk")

missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
if missing:
    # last-resort: pull from any leftover checkpoint
    cks = glob.glob(os.path.join(OUTPUT_ROOT, "**", "checkpoint-*", "adapter_model.safetensors"),
                    recursive=True)
    if cks:
        cdir = os.path.dirname(sorted(cks)[-1])
        os.makedirs(src_dir, exist_ok=True)
        for fn in needed:
            sp = os.path.join(cdir, fn)
            if os.path.exists(sp): shutil.copy2(sp, os.path.join(src_dir, fn))
    missing = [n for n in needed if not os.path.exists(os.path.join(src_dir, n))]
    if missing:
        raise FileNotFoundError(
            f"Adapter files {missing} missing in {src_dir}. Re-run the launch cell "
            f"(it saves the adapter), or free disk and retry.")

# patch adapter_config IN PLACE (no duplicate dir)
cfg_path = os.path.join(src_dir, "adapter_config.json")
with open(cfg_path) as f: cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(cfg_path, "w") as f: json.dump(cfg, f, indent=2)

# zip the two files DIRECTLY from src_dir -> WORKING (no 4 GB intermediate copy)
zip_path = os.path.join(WORKING, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fn in needed:
        zf.write(os.path.join(src_dir, fn), fn)
print(f"submission.zip -> {zip_path}  ({os.path.getsize(zip_path)/1024/1024:.1f} MB)  ready.")

[package] removed outputs/minmax_run (checkpoints) to free disk
submission.zip -> /kaggle/working/submission.zip  (3644.4 MB)  ready.
